**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/rozsa-2023"], check=True)
    os.chdir("/content/encore/entries/rozsa-2023")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Rozsa et al. (2023) on DANDI:000341, part 2: RAF induction recordings

Paper: Rozsa, Toth, Olah, Baka, Lakovics, Barzo and Tamas (2023), Sci. Adv. 9, eade4511,
doi:10.1126/sciadv.ade4511. Claims file: `claims/rozsa-2023.yaml`, written from the version of
record (Europe PMC full text) and its supplement.

Dataset: DANDI:000341, version 0.241220.1312 (doi:10.48324/dandi.000341/0.241220.1312), the only
published version; the paper cites the dandiset without a version.

This notebook covers the folders `Human_persistent_RAF` (313 files) and `Rat_persistent_RAF`
(132 files), the RAF induction recordings. Each file is taken as one cell (checked for repeats
below).

Revised after the first verification round: the threshold is found on a trace smoothed to the
Methods' 10 kHz filter, artefact APs are removed before any analysis, RA-APs must have no positive
current in absolute terms (holding current included), the mixture is fitted on robustly scaled
features, and each result is also shown without the drug-treated files and without the cells that
fire spontaneously before induction.

| Claim | Here |
|---|---|
| fig1ab-raf-patterns | tested |
| fig1c-ra-lower-threshold-slope | tested |
| fig1d-human-raf-proportion | tested |
| fig1f-human-raf-ngfc | blocked, metadata_insufficient (no cell class in the files) |
| figs1b-rat-raf-proportion | tested |
| figs1d-rat-raf-values | tested |
| fig1g-rat-raf-ngfc | blocked, metadata_insufficient |
| fig2e-sag-species | tested |
| fig2e-sag-raf-smaller | tested |
| fig2il-kcl-induces-ra-aps | blocked, data_missing (no KCl application in any file's metadata) |

In [ ]:
import zlib, time, resource, collections
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import raf

%matplotlib inline
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
PAPER = "rozsa-2023"


def seed(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())


T0 = time.time()
assets = raf.list_assets()
groups = collections.defaultdict(list)
for a in assets:
    groups[a["path"].split("/")[0]].append(a)
res = {}
for g in ("Human_persistent_RAF", "Rat_persistent_RAF"):
    t = time.time()
    res[g] = raf.process_group(groups[g], workers=4)
    print(f"{g}: {len(res[g])} files, {sum(a['size'] for a in groups[g]) / 1e9:.0f} GB in "
          f"{(time.time() - t) / 60:.1f} min; peak memory "
          f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

## Per-cell reduction

For each file: the induction sweeps (`raf.induction_sweeps`: shorter than 1 s, one positive step
of 50 to 300 ms), AIS-APs (APs during positive current steps) and RA-APs by reading (c) of the
claims file's header (`raf.ra_aps_induction`: APs after the first induction sweep with at most 5
pA of current at threshold, both relative to the holding level and in absolute terms, rebound APs
within 0.3 s of a hyperpolarizing step left out). Artefact APs (threshold outside -80 to -10 mV
or peak above 100 mV, `raf.plausible`) are removed first. `cont` marks the RA-APs of the
continuous recording (sweeps without any step), which is what the Methods count ("number of
RA-APs during continuous recording"). The baseline is the APs without current in the sweeps
before the first induction sweep (`raf.spontaneous_before`). Sag from the sweeps with one
hyperpolarizing step, taken before the first induction sweep where there are any.

In [ ]:
def drugs_of(m):
    return sorted({v for k, v in m["pharmacology"].items() if "Name" in k and v != "NaN"})


cells = []
for g, sp in (("Human_persistent_RAF", "human"), ("Rat_persistent_RAF", "rat")):
    for p, (m, aps, sw, _) in res[g].items():
        n_main = int(aps.main.sum()) if len(aps) else 0
        aps = aps[aps.main & raf.plausible(aps)] if len(aps) else aps
        ais, ra, t0, runs = raf.ra_aps_induction(aps, sw)
        _, ra_rel, _, _ = raf.ra_aps_induction(aps, sw, absolute=False)
        stepless = set(sw[sw.nsteps == 0].series)
        ra = ra.assign(cont=ra.series.isin(stepless)) if len(ra) else ra.assign(cont=[])
        n_pre, n_pre_cont, pre_s = raf.spontaneous_before(aps, sw, t0)
        sag = sw.dropna(subset=["sag_a"]) if "sag_a" in sw else sw.iloc[:0]
        if len(sag) and np.isfinite(t0) and (sag.start < t0).any():
            sag = sag[sag.start < t0]
        v = sw[sw.chan.str.startswith("Vm")].sort_values("start")
        post = v[(v.start >= t0) & (v.nsteps == 0)] if np.isfinite(t0) else v.iloc[:0]
        cells.append(dict(species=sp, file=p.split("/")[-1], subject=m["subject_subject_id"], meta=m,
                          aps=aps, ais=ais, ra=ra, t0=t0, runs=runs, sag=sag,
                          n_main=n_main, n_art=n_main - len(aps), n_ra_rel=len(ra_rel),
                          n_pre=n_pre, n_pre_cont=n_pre_cont, pre_s=pre_s,
                          hold_post=float(post.hold.median()) if len(post) else np.nan,
                          hold_post_max=float(post.hold.max()) if len(post) else np.nan,
                          post_s=float((post.n / post.rate).sum()),
                          drugs=drugs_of(m), ca=m["pharmacology"].get("Ca (mM)", "NaN"),
                          mg=m["pharmacology"].get("Mg (mM)", "NaN"),
                          n_ind=len(raf.induction_sweeps(sw)), n_sweeps=len(v),
                          fingerprint=(len(v), tuple(v.vmed.round(3).head(20))),
                          rates=sorted(set(v.rate.round())), n_stim_pA=int(sw.get("stim_in_pA", pd.Series([False])).sum()),
                          n_stim_bad=int(sw.get("stim_bad", pd.Series([False])).sum()),
                          n_nostim_aps=int(aps.i_thr.isna().sum()) if len(aps) else 0))
for c in cells:
    c["clean"] = not c["drugs"]
    c["quiet_before"] = c["n_pre"] == 0
print(len(cells), "cells")

## Data check

In [ ]:
C = pd.DataFrame([{k: v for k, v in c.items() if k in ("species", "file", "subject", "t0", "n_ind", "n_sweeps",
                                                       "n_stim_pA", "n_stim_bad", "n_main", "n_art", "n_nostim_aps",
                                                       "n_pre", "n_pre_cont", "pre_s", "hold_post", "hold_post_max",
                                                       "post_s", "clean", "ca", "mg")}
                  | {"n_ais": len(c["ais"]), "n_ra": len(c["ra"]), "n_ra_rel": c["n_ra_rel"], "n_sag": len(c["sag"])}
                  for c in cells])
for sp, d in C.groupby("species"):
    print(f"{sp}: files {len(d)}, subjects {d.subject.nunique()}, files with induction sweeps "
          f"{int((d.n_ind > 0).sum())}, with sag sweeps {int((d.n_sag > 0).sum())}; sweeps per file median "
          f"{d.n_sweeps.median():.0f}; stimulus stored in pA in {int((d.n_stim_pA > 0).sum())} files, "
          f"unusable in {int((d.n_stim_bad > 0).sum())}")
    print(f"   APs (peak > 0 mV) {d.n_main.sum()}, of which artefacts removed (threshold outside -80 to -10 mV "
          f"or peak above 100 mV) {d.n_art.sum()}; APs in sweeps without a stimulus series (never RA-APs or "
          f"AIS-APs) {d.n_nostim_aps.sum()}")
rates = collections.Counter(r for c in cells for r in c["rates"])
print("sampling rates present (files):", dict(rates))
allaps = pd.concat([c["aps"].assign(species=c["species"], cell=i) for i, c in enumerate(cells) if len(c["aps"])],
                   ignore_index=True)
print(f"APs kept: {len(allaps)}; threshold {allaps.thr.min():.1f} to {allaps.thr.max():.1f} mV, "
      f"peak {allaps.peak.min():.1f} to {allaps.peak.max():.1f} mV, slope {allaps.slope.min():.1f} to {allaps.slope.max():.1f} mV/ms")
fp = collections.defaultdict(list)
for c in cells:
    fp[c["fingerprint"]].append(c["file"])
print("files with identical sweep fingerprints (same sweep count and median Vm of 20 sweeps):",
      [v for v in fp.values() if len(v) > 1] or "none")
words = collections.Counter(d for c in cells for d in c["drugs"])
print("drugs named in general/pharmacology (files):", dict(words))
print("files with any drug named:", dict(C.assign(drug=~C.clean).groupby("species").drug.sum()))
print("Ca and Mg of the bath (mM), files:", dict(collections.Counter((c["species"], c["ca"], c["mg"]) for c in cells)))
kcl = [c["file"] for c in cells if any(("kcl" in str(v).lower() or "potassium" in str(v).lower() or "k+" in str(v).lower())
                                       for v in list(c["meta"]["pharmacology"].values()) + [c["meta"].get("session_description"), c["meta"].get("experiment_description")])]
print("files whose metadata mention KCl or potassium:", kcl or "none")
ctype = [c["file"] for c in cells if any(k.lower().startswith(("celltype", "anatomy")) for k in c["meta"]["pharmacology"])]
print("files with a cell class field:", len(ctype))
print("session descriptions:", dict(collections.Counter(c["meta"]["session_description"] for c in cells)))
print("subject descriptions:", dict(collections.Counter(str(c["meta"].get("subject_description")) for c in cells).most_common(8)))
# Session identifiers that also occur in the active-slice folders (the same cell deposited twice?).
import re
ses = lambda p: re.search(r"ses-([^_]+)_icephys", p).group(1)
active = {ses(a["path"]) for g in ("In_vitro_oscillation_Rat", "In_vitro_Human_elevated_neuronal_activity") for a in groups[g]}
shared = [c for c in cells if ses(c["file"]) in active]
for c in cells:
    c["in_active_folder"] = ses(c["file"]) in active
print(f"induction files whose session identifier also names an active-slice file: {len(shared)} "
      f"({sum(c['species'] == 'human' for c in shared)} human, {sum(c['species'] == 'rat' for c in shared)} rat); "
      f"with a drug named: {sum(not c['clean'] for c in shared)}")
size = collections.defaultdict(list)
for a in groups["Human_persistent_RAF"] + groups["Rat_persistent_RAF"]:
    size[a["size"]].append(a["path"].split("/")[-1])
same = [v for v in size.values() if len(v) > 1]
fpd = {c["file"]: c["fingerprint"] for c in cells}
print("induction files of identical size:", same or "none",
      "; identical fingerprints among them:", [v for v in same if len({fpd.get(f) for f in v}) == 1] or "none")

Holding current and spontaneous firing before induction (verifier, round 1). `hold_post` is the
median holding current of the stepless sweeps after the first induction sweep, `hold_post_max`
its maximum; `n_pre` the APs without current before the first induction sweep (`n_pre_cont` of
them in stepless sweeps), over `pre_s` seconds of stepless recording. `n_ra_rel` is the RA-AP count
when the current is taken relative to the holding level only (the rule of the first round), `n_ra`
with the absolute current also at most 5 pA.

In [ ]:
for sp, d in C.groupby("species"):
    t = d[d.n_ind > 0]
    print(f"{sp}: {len(t)} files with induction; holding current after induction (median per file) "
          f"{t.hold_post.min():.0f} to {t.hold_post.max():.0f} pA, median {t.hold_post.median():.0f}; files with a "
          f"positive holding current (> 5 pA) at any point after induction {int((t.hold_post_max > 5).sum())}; "
          f"RA-APs {t.n_ra_rel.sum()} by the relative rule, {t.n_ra.sum()} by the absolute rule")
    print(f"   files with spontaneous APs before induction {int((t.n_pre > 0).sum())} (in stepless sweeps "
          f"{int((t.n_pre_cont > 0).sum())}); stepless recording before induction median {t.pre_s.median():.0f} s; "
          f"pre-induction rate in those files median "
          f"{(t.n_pre_cont / t.pre_s.where(t.pre_s > 0))[t.n_pre_cont > 0].median():.3f} Hz")
with pd.option_context("display.max_rows", 500):
    print(C[C.n_ind > 0][["species", "file", "clean", "hold_post", "hold_post_max", "n_pre", "n_pre_cont", "pre_s",
                          "post_s", "n_ais", "n_ra_rel", "n_ra"]].round(1).to_string())

Threshold detection on a synthetic AP (a 1 mV/ms ramp, then an exponential upstroke whose dV/dt
crosses 5 mV/ms at a known time, Gaussian noise of 0.2 mV), at the sampling rates in the files.

In [ ]:
def synthetic(rate, noise, rng):
    t = np.arange(0, 0.05, 1 / rate)
    v = np.full_like(t, -65.0)
    r_ = (t > 0.02) & (t <= 0.03)
    v[r_] += (t[r_] - 0.02) * 1000
    v[t > 0.03] = -55.0
    k = (t > 0.03) & (t < 0.033)
    v[k] = np.minimum(-55 + np.cumsum(np.exp((t[k] - 0.03) * 1000 / 0.25)) / (rate / 1000), 30)
    v[t >= 0.033] = -60.0
    return v + rng.normal(0, noise, len(v)), 0.03 + 0.25e-3 * np.log(5)


rng = np.random.default_rng(seed("fig1c-ra-lower-threshold-slope"))
for rate in (5000, 16667, 25000, 50000, 100000, 200000):
    out = [raf._ap_table(synthetic(rate, 0.2, rng)[0], rate, None) for _ in range(30)]
    th = [o[0]["thr"] for o in out if o]
    sl = [o[0]["slope"] for o in out if o]
    print(f"{rate:6d} Hz: detected {len(th)}/30, threshold median {np.median(th):.2f} mV (true about -53.7), "
          f"slope median {np.median(sl):.2f} mV/ms")

Reading (b): one mixture per species on all its kept APs, fitted to a seeded draw of 200,000 of them
(`raf.gmm_classes`). The component means, SDs and weights are printed, with whether the RA
component (lower threshold) also has the lower slope, as the paper describes RA-APs.

In [ ]:
allaps["gmm_ra"] = False
GMM = {}
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax, sp in zip(axs, ("human", "rat")):
    idx = (allaps.species == sp).to_numpy()
    sd_ = seed("fig1ab-raf-patterns") + (0 if sp == "human" else 1)
    lab, info = raf.gmm_classes(allaps[idx], sd_)
    allaps.loc[idx, "gmm_ra"] = lab
    GMM[sp] = info
    print(sp, {k: (round(v, 3) if isinstance(v, float) else v) for k, v in info.items()})
    print(f"   RA component size under seeds +0, +1000 to +4000: {raf.gmm_seed_range(allaps[idx], sd_)}")
    d = allaps[idx]
    h = ax.hexbin(d.thr, d.slope.clip(-5, 15), gridsize=80, bins="log", extent=(-80, -10, -5, 15))
    ax.plot([info["thr_ra"], info["thr_ais"]], [info["slope_ra"], info["slope_ais"]], "r+", ms=14, mew=2)
    ax.set(title=f"{sp} induction APs (n = {len(d)}); + component means", xlabel="threshold (mV)",
           ylabel="pre-AP slope (mV/ms), clipped to -5 to 15")
    print(f"   slopes outside -5 to 15 mV/ms (clipped in the plot): {int((~d.slope.between(-5, 15)).sum())}")
plt.tight_layout()
plt.show()
for i, c in enumerate(cells):
    c["aps"] = c["aps"].assign(gmm=allaps.loc[allaps.cell == i, "gmm_ra"].to_numpy()) if len(c["aps"]) else c["aps"].assign(gmm=[])
    c["ra"] = c["ra"].assign(gmm=c["aps"].loc[c["ra"].index, "gmm"].to_numpy()) if len(c["ra"]) else c["ra"].assign(gmm=[])

In [ ]:
SUBSETS = {"all": lambda c: True, "drug-free": lambda c: c["clean"],
           "quiet before induction": lambda c: c["quiet_before"]}


def cell_epochs(c, reading, which):
    """RAF epochs of a cell. which: 'cont' (continuous recording, reading (c)), 'all' (with the
    interstimulus intervals, reading (c)), 'gmm' (continuous recording, RA-APs that the mixture
    also labels RA, reading (b)). reading: 'i' one epoch per cell, 'ii' one per induction run."""
    ra = c["ra"]
    if not len(ra):
        return []
    if which == "cont":
        ra = ra[ra.cont]
    elif which == "gmm":
        ra = ra[ra.cont & ra.gmm]
    return raf.epochs(ra.t_abs, None if reading == "i" else c["runs"])


def summarize(sp, reading, which, subset="all"):
    E = pd.DataFrame([e | {"file": c["file"]} for c in cells if c["species"] == sp and SUBSETS[subset](c)
                      for e in cell_epochs(c, reading, which)])
    if not len(E):
        return E, {}
    E["persistent"] = E.freq > 5
    P, S = E[E.persistent], E[~E.persistent]
    out = dict(n_epochs=len(E), n_persistent=len(P), n_sporadic=len(S),
               pers_n_median=P.n.median(), pers_freq_median=P.freq.median(), pers_dur_median=P.duration.median(),
               pers_dur_mean=P.duration.mean(), pers_dur_min=P.duration.min(), pers_dur_max=P.duration.max(),
               spor_n_median=S.n.median(), spor_freq_median=S.freq.median(),
               all_dur_median=E.duration.median(), all_dur_mean=E.duration.mean(),
               all_dur_min=E.duration.min(), all_dur_max=E.duration.max(),
               pers_n_sd=P.n.std(), pers_freq_sd=P.freq.std(), spor_n_sd=S.n.std(), spor_freq_sd=S.freq.std())
    return E, out


def judge(out, exp, keys):
    hits = []
    for k, (v, tol) in zip(keys, exp):
        x = out.get(k, np.nan)
        hits.append(bool(np.isfinite(x) and abs(x - v) <= tol))
    return hits

## fig1ab-raf-patterns

> Human layer 1 interneurons show two forms of retroaxonal firing (RAF) after induction,
> persistent RAF with barrages of RA-APs at high frequency (median 88 RA-APs, 13.9 Hz; duration
> median 17.1 s, mean 68.4 s, from 0.18 to 2809 s) and sporadic RAF with a few RA-APs at low
> frequency (median 8 RA-APs, 0.31 Hz).

Readings: epochs (i) one per cell, (ii) one per induction run; RA-APs of the continuous
recording by reading (c), with the interstimulus intervals included as a variant, and those the
mixture also labels RA by reading (b). The durations are judged under persistent RAF and shown
over all epochs. Each reading on all cells, on the drug-free files, and on the cells without
spontaneous APs before induction.

In [ ]:
EXP1 = [(88, 8.8), (13.9, 1.39), (17.1, 1.71), (68.4, 6.84), (0.18, 0.018), (2809, 280.9), (8, 0.8), (0.31, 0.031)]
K1 = ["pers_n_median", "pers_freq_median", "pers_dur_median", "pers_dur_mean", "pers_dur_min", "pers_dur_max",
      "spor_n_median", "spor_freq_median"]
rows = []
for subset in SUBSETS:
    for which in ("cont", "all", "gmm"):
        for reading in ("i", "ii"):
            E, out = summarize("human", reading, which, subset)
            h = judge(out, EXP1, K1)
            rows.append(dict(cells=subset, RA=which, epochs=reading, hits=sum(h),
                             **{k: out.get(k, np.nan) for k in ["n_epochs", "n_persistent", "n_sporadic"] + K1},
                             pers_n_sd=out.get("pers_n_sd"), spor_freq_sd=out.get("spor_freq_sd"),
                             all_dur_median=out.get("all_dur_median"), all_dur_mean=out.get("all_dur_mean"),
                             all_dur_min=out.get("all_dur_min"), all_dur_max=out.get("all_dur_max")))
T1 = pd.DataFrame(rows)
with pd.option_context("display.max_columns", 40, "display.width", 250):
    print(T1.round(3).to_string())
print("expected:", dict(zip(K1, [e[0] for e in EXP1])))
print("most values within tolerance under any reading:", T1.hits.max())

**Result: not reproduced.** Persistent and sporadic epochs both occur under every reading on all
cells, but at most two of the eight values are within tolerance under any reading or subset (the
cells quiet before induction, RA-APs with the interstimulus intervals, one epoch per run: maximum
duration 2616 s and sporadic frequency 0.294 Hz). The first reading (continuous recording, one
epoch per cell, all cells) has none. The mixture of reading (b) fails its check (its RA component
has the steeper mean slope, about 6.9 against 1.9 mV/ms in human, and its size changes with the seed
from about 88,000 to about 2,450,000 APs), so the `gmm` rows are not a reading of the paper's classes.

## fig1c-ra-lower-threshold-slope

> RA-APs have a lower somatic threshold potential and a lower pre-AP depolarization slope than
> AIS-APs, during persistent and during sporadic RAF.

Reading (c): AIS-APs during the current pulses, RA-APs after induction (interstimulus intervals
and continuous recording), within the cells whose one epoch (reading (i), RA-APs of the
continuous recording) is persistent or sporadic. APs pooled; the paper's rule (KS normality,
then t test or Mann-Whitney U). Per-cell medians as a variant. Then the same comparison within
each sampling rate, since the rates differ between the protocols.

In [ ]:
SEL = {}
for sp in ("human", "rat"):
    for pattern in ("persistent", "sporadic"):
        A, B, PA, PB = [], [], [], []
        for c in cells:
            if c["species"] != sp:
                continue
            ep = cell_epochs(c, "i", "cont")
            if not ep:
                continue
            pers = ep[0]["freq"] > 5
            if pers != (pattern == "persistent"):
                continue
            A.append(c["ais"]); B.append(c["ra"])
            PA.append(c["ais"][["thr", "slope"]].median()); PB.append(c["ra"][["thr", "slope"]].median())
        if not A:
            print(sp, pattern, "no cells"); continue
        A, B = pd.concat(A), pd.concat(B)
        SEL[(sp, pattern)] = (A, B)
        PA, PB = pd.DataFrame(PA), pd.DataFrame(PB)
        for q in ("thr", "slope"):
            t, p = raf.paper_test(B[q], A[q])
            tc = stats.mannwhitneyu(PB[q].dropna(), PA[q].dropna()).pvalue if len(PB[q].dropna()) and len(PA[q].dropna()) else np.nan
            print(f"{sp:5s} {pattern:10s} {q:5s}: cells {len(PA)}; AIS-AP median {A[q].median():7.2f} (n={len(A)}), "
                  f"RA-AP median {B[q].median():7.2f} (n={len(B)}); {t} P = {p:.2g}; per-cell medians "
                  f"{PA[q].median():.2f} vs {PB[q].median():.2f}, Mann-Whitney P = {tc:.2g}; "
                  f"cells with RA-AP median below AIS-AP median {int((PB[q] < PA[q]).sum())} of {int((PB[q].notna() & PA[q].notna()).sum())}")
print()
print("By sampling rate (human and rat, both patterns pooled): median threshold (mV) and slope (mV/ms)")
for sp in ("human", "rat"):
    A = pd.concat([SEL[k][0] for k in SEL if k[0] == sp]).assign(cls="AIS")
    B = pd.concat([SEL[k][1] for k in SEL if k[0] == sp]).assign(cls="RA")
    D = pd.concat([A, B])
    D["rate_khz"] = (D.rate / 1000).round(1)
    t = D.groupby(["rate_khz", "cls"]).agg(n=("thr", "size"), thr=("thr", "median"), slope=("slope", "median")).unstack("cls")
    print(sp)
    print(t.round(2).to_string())

**Result: reproduced.** In the 24 human persistent cells RA-APs have a lower threshold (median
-59.6 against -44.1 mV) and slope (0.77 against 1.79 mV/ms), and in the 119 sporadic cells too
(-61.4 against -44.8 mV, 0.74 against 2.18 mV/ms), with Mann-Whitney P printed as 0 (below double
precision) and per-cell P from 1.4e-16 to 0.0006. The order holds within each human sampling rate
that has both classes (10 to 100 kHz).

## fig1d-human-raf-proportion and figs1b-rat-raf-proportion

> About one-fourth of tested human layer 1 interneurons (87 of 385) show RAF. About a third of
> tested rat layer 1 interneurons (102 of 259) show RAF.

A cell shows RAF when at least one RA-AP occurs after induction. Tested cells are the files with
induction sweeps; all files are shown as the other denominator. RA-APs by reading (c) (with or
without the interstimulus intervals) and by reading (b) (reading (c) RA-APs the mixture also
labels RA). At least two RA-APs is shown as a variant against single spurious events. Subsets:
all tested cells; drug-free files; cells with spontaneous APs before induction counted as without
RAF, or left out.

In [ ]:
prop = []
for c in cells:
    ra = c["ra"]
    prop.append(dict(species=c["species"], file=c["file"], tested=c["n_ind"] > 0, clean=c["clean"],
                     quiet=c["quiet_before"], c_all=len(ra), c_cont=int(ra.cont.sum()) if len(ra) else 0,
                     b_all=int(ra.gmm.sum()) if len(ra) else 0))
prop = pd.DataFrame(prop)
PROP = []
for sp, exp in (("human", 0.226), ("rat", 0.394)):
    d = prop[prop.species == sp]
    for subset in ("all tested", "drug-free", "firing before induction = no RAF", "firing before induction left out"):
        t = d[d.tested]
        if subset == "drug-free":
            t = t[t.clean]
        if subset == "firing before induction left out":
            t = t[t.quiet]
        for col, name in (("c_all", "reading (c), ISIs and continuous"), ("c_cont", "reading (c), continuous only"),
                          ("b_all", "reading (b)")):
            for k in (1, 2):
                has = t[col] >= k
                if subset == "firing before induction = no RAF":
                    has = has & t.quiet
                n = int(has.sum())
                PROP.append(dict(species=sp, cells=subset, RA=name, min_ra=k, n=n, of=len(t), fraction=n / max(len(t), 1),
                                 within=abs(n / max(len(t), 1) - exp) <= exp * 0.1))
PROP = pd.DataFrame(PROP)
print(PROP.round(3).to_string())
print("expected human 0.226, rat 0.394 (tolerance 10%); denominator with all files: human "
      f"{int((prop.species == 'human').sum())}, rat {int((prop.species == 'rat').sum())}")

**Result (fig1d-human-raf-proportion): not reproduced.** With at least one RA-AP, as the criterion
says, 301 of 307 human cells (0.980) have RAF by reading (c) and 159 (0.518) on the continuous
recording alone; the drug-free files give 0.976 and 0.446, and counting the cells that fire before
induction as without RAF gives 0.586 and 0.280. Only variants outside the criterion fall within
tolerance: at least two continuous-recording RA-APs with the cells firing before induction counted
as without RAF (0.241), and a reading (b) row whose mixture fails its check (0.238).

**Result (figs1b-rat-raf-proportion): not reproduced.** 115 of 118 rat cells (0.975) have RA-APs
by reading (c) and 97 (0.822) on the continuous recording; the lowest fraction in the table is
0.500, against 0.394.

## figs1d-rat-raf-values

> Rat interneurons show both sporadic RAF (frequency 1 Hz, 7 RA-APs) and tonic RAF (frequency
> 6.9 Hz, 100 RA-APs).

In [ ]:
EXP2 = [(1, 0.1), (7, 0.7), (6.9, 0.69), (100, 10)]
K2 = ["spor_freq_median", "spor_n_median", "pers_freq_median", "pers_n_median"]
rows = []
for subset in SUBSETS:
    for which in ("cont", "all", "gmm"):
        for reading in ("i", "ii"):
            E, out = summarize("rat", reading, which, subset)
            rows.append(dict(cells=subset, RA=which, epochs=reading, hits=sum(judge(out, EXP2, K2)),
                             **{k: out.get(k, np.nan) for k in ["n_epochs", "n_persistent", "n_sporadic"] + K2}))
T2 = pd.DataFrame(rows)
print(T2.round(3).to_string())
print("expected:", dict(zip(K2, [e[0] for e in EXP2])))
print("most values within tolerance under any reading:", T2.hits.max())

**Result: not reproduced.** No rat value is within tolerance under reading (c) in any subset; one
reading (b) row has one of four, and that mixture fails its check.

## fig2e-sag-species and fig2e-sag-raf-smaller

> Human layer 1 interneurons have a higher somatic sag fraction than rat ones (0.15 against
> 0.04). In both species, interneurons with RAF have smaller somatic sag fractions than those
> without (human 0.11 against 0.16; rat 0.02 against 0.06).

Per cell, the sag of one step sweep: the most hyperpolarizing step, or the step whose peak is
nearest -90 mV; formula (a) or (b) (`raf._sag`). RAF status (at least one RA-AP) by each reading
of fig1d: reading (c) with the interstimulus intervals, reading (c) on the continuous recording
only, and reading (b); over the tested cells with sag sweeps (the denominator of fig1d) and the
drug-free ones among them. All files with sag sweeps, untested ones counted as without RAF, are
shown for comparison.

In [ ]:
sg = []
for c, pr in zip(cells, prop.itertuples()):
    s = c["sag"]
    if not len(s):
        continue
    most = s[s.i_step == s.i_step.min()].iloc[0]
    near = s.iloc[int(np.argmin(np.abs(s.vpeak.to_numpy() + 90)))]
    sg.append(dict(species=c["species"], file=c["file"], clean=c["clean"], raf_c_all=pr.c_all > 0,
                   raf_c_cont=pr.c_cont > 0, raf_b=pr.b_all > 0, tested=pr.tested,
                   a_most=most.sag_a, b_most=most.sag_b, a_90=near.sag_a, b_90=near.sag_b,
                   step_most=most.i_step, peak_most=most.vpeak))
sg = pd.DataFrame(sg)
print(sg.groupby("species")[["a_most", "b_most", "a_90", "b_90", "step_most", "peak_most"]].describe().T.round(3))

In [ ]:
for col in ("a_most", "b_most", "a_90", "b_90"):
    for label, d in (("all", sg), ("drug-free", sg[sg.clean])):
        h, r = d[d.species == "human"][col].dropna(), d[d.species == "rat"][col].dropna()
        p = stats.mannwhitneyu(h, r).pvalue
        print(f"{col:7s} {label:9s}: human n={len(h)} median {h.median():.3f} (SD {h.std():.3f}); rat n={len(r)} median "
              f"{r.median():.3f} (SD {r.std():.3f}); Mann-Whitney P = {p:.2g}")
print("expected: human 0.15 (n=358), rat 0.04 (n=254), tolerance 10%, P < 0.001")

**Result: partially reproduced.** Human sag is higher than rat under every formula and step (0.198
against 0.071, Mann-Whitney P = 1.5e-26, formula (a) on the most hyperpolarizing step; drug-free
files 0.211 against 0.082, P = 8e-10), but neither median is within tolerance of 0.15 and 0.04.

In [ ]:
EXP3 = [0.11, 0.16, 0.02, 0.06]
rows = []
# RAF status is that of fig1d, whose denominator is the tested cells (files with induction sweeps);
# all files with sag sweeps, which counts the untested ones as without RAF, is shown for comparison
# (verifier, round 2).
CELLS3 = {"tested": sg[sg.tested], "tested, drug-free": sg[sg.tested & sg.clean],
          "all files with sag": sg}
print("files with sag sweeps but no induction sweeps:", sg[~sg.tested].groupby("species").size().to_dict())
for status in ("raf_c_all", "raf_c_cont", "raf_b"):
    for label, d0 in CELLS3.items():
        for col in ("a_most", "b_most", "a_90", "b_90"):
            row = dict(status=status, cells=label, sag=col)
            meds = []
            for sp in ("human", "rat"):
                d = d0[d0.species == sp]
                y, n = d[d[status]][col].dropna(), d[~d[status]][col].dropna()
                p = stats.mannwhitneyu(y, n).pvalue if len(y) and len(n) else np.nan
                row |= {f"{sp}_raf_n": len(y), f"{sp}_raf": y.median(), f"{sp}_noraf_n": len(n),
                        f"{sp}_noraf": n.median(), f"{sp}_P": p,
                        f"{sp}_lower_p001": bool(y.median() < n.median() and p < 0.001)}
                meds += [y.median(), n.median()]
            row["medians_within"] = int(sum(abs(m - e) <= 0.1 * e for m, e in zip(meds, EXP3) if np.isfinite(m)))
            rows.append(row)
T3 = pd.DataFrame(rows)
with pd.option_context("display.max_columns", 40, "display.width", 250):
    print(T3.to_string(float_format=lambda x: f"{x:.3g}"))
print("expected: human 0.11 (n=83) vs 0.16 (n=275); rat 0.02 (n=97) vs 0.06 (n=157); P < 0.001")
print(f"notebook time {(time.time() - T0) / 60:.1f} min, peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

**Result: partially reproduced, resting on the RAF reading.** On the tested cells, with RAF status
from the RA-APs of the continuous recording (reading (c) without the interstimulus intervals), rat
cells with RAF have the lower sag at P < 0.001 (formula (a), most hyperpolarizing step: 0.060, n =
95, against 0.125, n = 20, Mann-Whitney P = 0.00039), while human cells do not (0.186 against 0.211,
P = 0.12). The rat result is fragile: the step nearest -90 mV gives P = 0.0080 and the drug-free
files P = 0.0061. Counting the files without induction sweeps as without RAF, as the round-2 version
did, gives rat P = 2.8e-05. With the interstimulus intervals neither species reaches P < 0.001 (rat
3 cells without RAF, P = 0.036; human 6, P = 0.54). No reading has more than one of the four medians
within tolerance. Reading (b) gives the human difference at P < 0.001, but its mixture fails its
check. The continuous-recording reading is the assumption: the Methods count RA-APs "during
continuous recording" and start that recording once APs appear in the interstimulus interval
(Methods, "Analysis of RAF patterns" and "Induction protocol of RAF").


## Summary

| Claim | Outcome |
|---|---|
| fig1ab-raf-patterns | not reproduced |
| fig1c-ra-lower-threshold-slope | reproduced |
| fig1d-human-raf-proportion | not reproduced |
| fig1f-human-raf-ngfc | blocked, metadata_insufficient |
| figs1b-rat-raf-proportion | not reproduced |
| figs1d-rat-raf-values | not reproduced |
| fig1g-rat-raf-ngfc | blocked, metadata_insufficient |
| fig2e-sag-species | partially reproduced |
| fig2e-sag-raf-smaller | partially reproduced (assumed) |
| fig2il-kcl-induces-ra-aps | blocked, data_missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. Induction sweeps, AIS-APs and RA-APs are rebuilt from
the stimulus and the traces (reading (c) of the claims file), as the files carry no labels;
epochs are one per cell or one per induction run, as the claims file names. Sag is taken on the
step sweeps before induction with a formula the paper does not give (two readings). Each result
is also shown on the drug-free files and on the cells that do not fire before induction. The
two-component mixture of reading (b) does not recover the paper's classes in these recordings
(its lower-threshold component has the steeper slope, and its size depends on the seed), so no
outcome rests on it.

What was found about the dataset. The folders hold 313 human and 132 rat files against 385 and 259
tested cells, and nearly every cell in them fires APs without current after induction, so they
appear to be a selection the dandiset does not describe. 124 human and 33 rat cells fire
spontaneous APs before induction. 58 human and 94 rat files list drugs or a changed calcium
concentration. No file names a cell class or a KCl application. Sampling rates range from 5 to
200 kHz.